# AI Code Mentor
**Automated Code Review with RAG + Static Analysis + LLM**

This notebook combines:
- **RAG**: Retrieves relevant PEP 8 rules from documentation
- **Static Analysis**: Runs Pylint for syntax/style errors
- **LLM**: CodeLlama-7B for deep logical analysis

---

## 1. Installation & Setup

In [ ]:
# Install all dependencies
%pip install -q -U bitsandbytes accelerate transformers pylint \
    langchain langchain-community langchain-core \
    langchain-text-splitters langchain-huggingface \
    faiss-cpu rank_bm25 beautifulsoup4 huggingface_hub \
    sentence-transformers streamlit pyngrok

In [ ]:
# Authentication
import os
from huggingface_hub import login

# Try Kaggle Secrets first, then environment variable
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    hf_token = os.getenv("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("[OK] Logged in to Hugging Face")
else:
    print("[WARNING] No HF_TOKEN found. Add it to Kaggle Secrets.")

---
## 2. Core Classes

In [ ]:
import torch
import json
import re
import sys
import os
import subprocess
import tempfile
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# LangChain Imports
from langchain_community.document_loaders import WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

try:
    from langchain_text_splitters import RecursiveCharacterTextSplitter
except ImportError:
    from langchain.text_splitter import RecursiveCharacterTextSplitter

print("[OK] All imports successful")

In [ ]:
# ================================================================
# KNOWLEDGE BASE - RAG Engine for PEP 8 Documentation
# ================================================================

class KnowledgeBase:
    """
    RAG-based knowledge retrieval from PEP 8 documentation.
    Uses hybrid search: FAISS (semantic) + BM25 (keyword).
    """
    
    def __init__(self, url="https://peps.python.org/pep-0008/"):
        print(f"[INFO] Ingesting Knowledge from: {url}")
        print("[1/4] Loading web content...")
        
        try:
            loader = WebBaseLoader(url)
            data = loader.load()
        except Exception as e:
            print(f"[WARNING] Web Load Warning: {e}. Using Internal Backup Rules.")
            backup_text = """
            Function Names: Function names should be lowercase, with words separated by underscores.
            Class Names: Class names should normally use the CapWords convention.
            Imports: Imports should usually be on separate lines and at the top of the file.
            Whitespace: Avoid extraneous whitespace immediately inside parentheses.
            Constants: Constants should be defined in all capital letters.
            """
            data = [Document(page_content=backup_text)]

        print("[2/4] Web content loaded. Splitting text...")
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        splits = text_splitter.split_documents(data)
        
        print("[3/4] Text split complete. Loading embedding model (this may take a minute)...")
        # CPU Embedding (Saves GPU for the LLM)
        embedding_model = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2",
            model_kwargs={'device': 'cpu'}
        )

        print("[3.5/4] Embedding model loaded. Creating vector store...")
        self.vectorstore = FAISS.from_documents(splits, embedding_model)
        self.bm25_retriever = BM25Retriever.from_documents(splits)
        self.bm25_retriever.k = 3
        
        print("[OK] Knowledge Base Ready.")

    def search(self, queries):
        """Hybrid search: combines semantic (FAISS) and keyword (BM25) results."""
        results = []
        for q in queries:
            docs_faiss = self.vectorstore.similarity_search(q, k=2)
            docs_bm25 = self.bm25_retriever.invoke(q)
            results.extend(docs_faiss + docs_bm25)
        
        unique_docs = {doc.page_content: doc for doc in results}.values()
        return "\n\n".join([f"[Guide Excerpt]: {doc.page_content}" for doc in unique_docs])

In [ ]:
# ================================================================
# UNIFIED CODE REVIEWER - RAG + Pylint + LLM
# ================================================================

class UnifiedCodeReviewer:
    """
    Main code review engine combining:
    - Static Analysis (Pylint)
    - RAG-based style guide lookup
    - LLM-powered deep analysis
    """
    
    def __init__(self, model_id="codellama/CodeLlama-7b-Instruct-hf"):
        # 1. Initialize RAG
        self.kb = KnowledgeBase()
        
        print(f"[INFO] Loading Brain: {model_id}...")
        print("[STEP 1/3] Initializing tokenizer...")
        
        # 2. Initialize Model (GPU with 4-bit quantization)
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )

        self.tokenizer = AutoTokenizer.from_pretrained(model_id)
        print("[STEP 2/3] Tokenizer loaded. Loading model with 4-bit quantization (this takes 2-3 minutes)...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            quantization_config=bnb_config,
            device_map="auto"
        )
        print("[STEP 3/3] Model loaded successfully!")
        print("[OK] System Online.")

    def _ask_llm(self, prompt, max_tokens=200):
        """Generate response from the LLM."""
        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")
        outputs = self.model.generate(
            **inputs, 
            max_new_tokens=max_tokens,
            temperature=0.1
        )
        return self.tokenizer.decode(outputs[0], skip_special_tokens=True).split("[/INST]")[-1].strip()

    def _run_pylint(self, code_snippet):
        """Run static analysis on the code using Pylint."""
        with tempfile.NamedTemporaryFile(mode='w+', suffix='.py', delete=False) as f:
            f.write(code_snippet)
            fname = f.name
            
        try:
            result = subprocess.run(
                ["pylint", fname, "--output-format=json"], 
                capture_output=True, text=True
            )
            
            if not result.stdout.strip(): 
                return "No Syntax Errors Found."
            
            try:
                errors = json.loads(result.stdout)
                if len(errors) > 15:
                    report = [f"Line {e['line']}: {e['message']} ({e['symbol']})" for e in errors[:15]]
                    report.append("... (and more)")
                else:
                    report = [f"Line {e['line']}: {e['message']} ({e['symbol']})" for e in errors]
                return "\n".join(report)
            except json.JSONDecodeError: 
                return "Pylint ran but output was not parseable."
        except Exception as e:
            return f"Pylint Failed to Run: {e}"
        finally:
            if os.path.exists(fname): 
                os.remove(fname)

    def _generate_search_plan(self, code_snippet):
        """Analyze code to decide what style topics to search for."""
        prompt = f"""<s>[INST] <<SYS>>
        You are a Technical Analyst.
        Identify 3 SPECIFIC Python Style topics relevant to the code provided.
        Output valid JSON only.
        {{ "queries": ["topic 1", "topic 2", "topic 3"] }}
        <</SYS>>
        Code: {code_snippet}
        JSON: [/INST]"""
        
        raw_response = self._ask_llm(prompt, max_tokens=150)
        
        try:
            json_match = re.search(r'\{.*\}', raw_response, re.DOTALL)
            if json_match:
                data = json.loads(json_match.group(0))
                return data.get("queries", [])
            else:
                return [raw_response.strip()]
        except json.JSONDecodeError:
            return ["General Python Style"]

    def review(self, code_snippet):
        """
        Main entry point: performs comprehensive code review.
        Returns formatted feedback string.
        """
        print("1. Analyzing Code Logic...")
        linter_report = self._run_pylint(code_snippet)
        
        print("2. Retrieving Style Rules...")
        search_terms = self._generate_search_plan(code_snippet)
        rag_rules = self.kb.search(search_terms)
        
        print("3. Generating Final Report...")
        final_prompt = f"""<s>[INST] <<SYS>>
        You are a Senior Software Architect and Python Expert.
        Conduct a comprehensive code review using the provided reports.
        
        INPUTS:
        1. STATIC ANALYSIS REPORT (Facts):
        {linter_report}
        
        2. STYLE GUIDE RULES (Reference):
        {rag_rules}
        
        CRITICAL ARCHITECTURAL PRINCIPLES (Only Mention Relevant Ones): 
        1. **Immutability & Safety:** Prefer creating new data structures over mutating existing ones.
        2. **Resource Management:** Ensure all resources (files, connections) are properly managed.
        3. **Algorithmic Efficiency:** Identify unnecessary complexity (e.g., nested loops that could be maps/sets).
        4. **Style Compliance:** Verify naming and formatting against the provided Style Guide Rules.
        
        OUTPUT FORMAT:
        **1. Critical Issues:** (Logical bugs, Safety risks, and Pylint errors)
        **2. Style Analysis:** (Comparison against the provided PEP 8 rules)
        **3. Refactored Solution:** (The robust, pythonic, and safe code)
        <</SYS>>

        User Code:
        {code_snippet} [/INST]"""
        
        result = self._ask_llm(final_prompt, max_tokens=1024)
        
        # Cleanup GPU memory
        gc.collect()
        torch.cuda.empty_cache()
        
        return result

---
## 3. Initialize the Engine
This loads the model (~1-2 minutes on first run)

In [ ]:
# Memory optimization
print("="*60)
print("INITIALIZATION STARTING")
print("="*60)
print("[INIT 1/2] Clearing GPU memory...")
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
gc.collect()
torch.cuda.empty_cache()
print("[INIT 1/2] GPU memory cleared.")

print("[INIT 2/2] Creating UnifiedCodeReviewer (this loads Knowledge Base + LLM)...")
engine = UnifiedCodeReviewer()
print("="*60)
print("INITIALIZATION COMPLETE!")
print("="*60)


---
## 4. Test Examples
Run these cells to test the reviewer on different code samples.

In [ ]:
# Test 1: Logic Bug (Modifying list while iterating)
code_logic_bug = """
def Process_List_Data(InputList):
    
    for item in InputList:
        if item % 2 == 0:
            InputList.remove(item)
    return InputList
"""

print("=" * 50)
print("TEST: Logic Bug (List Mutation)")
print("=" * 50)
print(engine.review(code_logic_bug))

In [ ]:
# Test 2: Style Issues
code_style = """
import os, sys, json 
def Do_Thing(x,y):
    z=x+y
    return z
"""

print("=" * 50)
print("TEST: Style Issues")
print("=" * 50)
print(engine.review(code_style))

In [ ]:
# Test 3: Performance Issues (O(n^2) complexity)
code_performance = """
def find_duplicates(items):
    dupes = []
    for i in range(len(items)):
        for j in range(len(items)):
            if i != j and items[i] == items[j]:
                dupes.append(items[i])
    return dupes
"""

print("=" * 50)
print("TEST: Performance Issues")
print("=" * 50)
print(engine.review(code_performance))

In [ ]:
# Test 4: Safety Issues (Mutable Default Argument)
code_safety = """
def add_student(name, student_list=[]):
    student_list.append(name)
    return student_list
"""

print("=" * 50)
print("TEST: Safety Issues")
print("=" * 50)
print(engine.review(code_safety))

In [ ]:
# Test 5: Perfect Code (should get minimal feedback)
code_perfect = """
def calculate_circle_area(radius):
    if radius < 0:
        raise ValueError("Radius cannot be negative")
    pi = 3.14159
    return pi * (radius ** 2)
"""

print("=" * 50)
print("TEST: Perfect Code")
print("=" * 50)
print(engine.review(code_perfect))

---
## 5. Streamlit Web App (Optional)
Run this section to launch the web interface via ngrok tunnel.

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import json
import re
import os
import subprocess
import tempfile
import gc

from langchain_community.document_loaders import WebBaseLoader
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter


class KnowledgeBase:
    def __init__(self, url="https://peps.python.org/pep-0008/"):
        try:
            loader = WebBaseLoader(url)
            data = loader.load()
        except Exception:
            backup_text = """
            Function Names: Function names should be lowercase, with words separated by underscores.
            Class Names: Class names should normally use the CapWords convention.
            Imports: Imports should usually be on separate lines and at the top of the file.
            Whitespace: Avoid extraneous whitespace immediately inside parentheses.
            Constants: Constants should be defined in all capital letters.
            """
            data = [Document(page_content=backup_text)]
        
        print("[2/4] Web content loaded. Splitting text...")
        text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
        splits = text_splitter.split_documents(data)
        
        embedding_model = HuggingFaceEmbeddings(
            model_name="all-MiniLM-L6-v2", 
            model_kwargs={'device': 'cpu'}
        )
        print("[3.5/4] Embedding model loaded. Creating vector store...")
        self.vectorstore = FAISS.from_documents(splits, embedding_model)
        self.bm25_retriever = BM25Retriever.from_documents(splits)
        self.bm25_retriever.k = 3

    def search(self, queries):
        results = []
        for q in queries:
            results.extend(self.vectorstore.similarity_search(q, k=2))
            results.extend(self.bm25_retriever.invoke(q))
        unique = {doc.page_content: doc for doc in results}.values()
        return "\n\n".join([f"[Guide]: {doc.page_content}" for doc in unique])


@st.cache_resource
def get_engine():
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from huggingface_hub import login
    
    hf_token = os.getenv("HF_TOKEN")
    if not hf_token:
        try:
            from kaggle_secrets import UserSecretsClient
            hf_token = UserSecretsClient().get_secret("HF_TOKEN")
        except Exception:
            pass
    
    if hf_token:
        login(token=hf_token)

    model_id = "codellama/CodeLlama-7b-Instruct-hf"
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id, 
        quantization_config=bnb_config, 
        device_map="auto"
    )
    model.kb = KnowledgeBase()
    return tokenizer, model


def generate_review(code_snippet):
    tokenizer, model = get_engine()
    
    # Static Analysis
    with tempfile.NamedTemporaryFile(mode='w+', suffix='.py', delete=False) as f:
        f.write(code_snippet)
        fname = f.name
    try:
        res = subprocess.run(
            ["pylint", fname, "--output-format=json"], 
            capture_output=True, text=True
        )
        if not res.stdout.strip(): 
            linter_report = "No Syntax Errors."
        else: 
            errors = json.loads(res.stdout)
            if len(errors) > 15:
                linter_report = "\n".join([f"Line {e['line']}: {e['message']}" for e in errors[:15]]) + "\n... (and more)"
            else:
                linter_report = "\n".join([f"Line {e['line']}: {e['message']}" for e in errors])
    except Exception:
        linter_report = "Pylint Failed."
    finally:
        if os.path.exists(fname): 
            os.remove(fname)

    # Dynamic Planning
    plan_prompt = f"""<s>[INST] <<SYS>>
    You are a Technical Analyst.
    Identify 3 SPECIFIC Python Style topics relevant to the code provided.
    Output valid JSON only. {{ "queries": ["topic 1", "topic 2", "topic 3"] }}
    <</SYS>>
    Code: {code_snippet}
    JSON: [/INST]"""
    
    inputs = tokenizer(plan_prompt, return_tensors="pt").to("cuda")
    raw_plan = tokenizer.decode(
        model.generate(**inputs, max_new_tokens=150)[0], 
        skip_special_tokens=True
    ).split("[/INST]")[-1]
    
    try:
        queries = json.loads(re.search(r'\{.*\}', raw_plan, re.DOTALL).group(0)).get("queries", [])
    except Exception:
        queries = ["General Python Style"]
    
    rag_rules = model.kb.search(queries)

    # Final Review
    final_prompt = f"""<s>[INST] <<SYS>>
    You are a Senior Software Architect.
    Conduct a comprehensive code review using the provided reports.
    
    INPUTS:
    1. STATIC ANALYSIS REPORT: {linter_report}
    2. STYLE GUIDE RULES: {rag_rules}
    
    CRITICAL INSTRUCTIONS:
    1. **Summarize Style:** Do NOT list every single style error. Group them.
    2. **Logic Safety:** Detect "Concurrent Modification" (modifying a list while iterating).
    3. **Algorithmic Efficiency:** Optimize O(n^2) nested loops using Sets/Dictionaries.
    4. **Refactored Code:** You MUST wrap the solution in ```python ... ``` blocks.
    
    OUTPUT FORMAT:
    **1. Critical Issues:**
    (List logical bugs and safety risks)
    
    **2. Style Analysis:**
    (Summarize PEP 8 issues)
    
    **3. Refactored Solution:**
    ```python
    (WRITE THE FULL CORRECTED CODE HERE)
    ```
    <</SYS>>

    User Code:
    {code_snippet} [/INST]"""
    
    inputs = tokenizer(final_prompt, return_tensors="pt").to("cuda")
    
    result = tokenizer.decode(
        model.generate(**inputs, max_new_tokens=2048, repetition_penalty=1.2)[0], 
        skip_special_tokens=True
    ).split("[/INST]")[-1]
    
    gc.collect()
    torch.cuda.empty_cache()
    
    return result


# UI
st.set_page_config(page_title="AI Code Mentor", layout="wide")
st.title("AI Code Mentor")
st.markdown("### Automated Review with RAG + Static Analysis")

col1, col2 = st.columns(2)

with col1:
    st.subheader("Input Code")
    code_input = st.text_area("Paste Python code here:", height=400, value="")
    
    if st.button("Run Review", type="primary"):
        if code_input.strip():
            with st.spinner("Analyzing Logic & Style..."):
                result = generate_review(code_input)
                st.session_state['result'] = result
        else:
            st.warning("Please enter some code first.")

with col2:
    st.subheader("Analysis Report")
    if 'result' in st.session_state:
        full_text = st.session_state['result']
        
        code_match = re.search(r"```(?:python)?\s*(.*?)```", full_text, re.DOTALL)
        
        if code_match:
            clean_code = code_match.group(1)
            text_part = full_text.replace(code_match.group(0), "")
            st.markdown(text_part)
            st.subheader("Refactored Code")
            st.code(clean_code, language='python')
        else:
            st.markdown(full_text)
            st.warning("No Code Block Detected. Check the text above.")
    else:
        st.info("Run a review to see results here.")

In [ ]:
# Launch Streamlit with ngrok tunnel
import time
import subprocess
from pyngrok import ngrok

# Cleanup
print("Cleaning up old processes...")
!pkill -9 streamlit
ngrok.kill()

# Set ngrok token (get from Kaggle Secrets or environment)
ngrok_token = None
try:
    from kaggle_secrets import UserSecretsClient
    ngrok_token = UserSecretsClient().get_secret("NGROK_TOKEN")
except Exception:
    ngrok_token = os.getenv("NGROK_TOKEN")

if ngrok_token:
    ngrok.set_auth_token(ngrok_token)
else:
    print("[WARNING] No NGROK_TOKEN found. Add it to Kaggle Secrets for tunneling.")

# Start Streamlit
print("Starting Streamlit App...")
process = subprocess.Popen(
    ["nohup", "streamlit", "run", "app.py", "&"], 
    stdout=subprocess.DEVNULL, 
    stderr=subprocess.DEVNULL
)

print("Waiting for server to initialize...")
time.sleep(10)

# Open tunnel
try:
    public_url = ngrok.connect(8501).public_url
    print(f"[OK] App is Live! Click here: {public_url}")
except Exception as e:
    print(f"[ERROR] Tunnel Error: {e}")

---
## 6. Memory Cleanup

In [ ]:
# Clear GPU memory when done
gc.collect()
torch.cuda.empty_cache()
print("[OK] GPU memory cleared")